# Task 1 — DPO (ATML PA2)
Runtime → Change runtime type → **GPU**. Run cells top to bottom.

**Disconnect-safe:** `outputs/` and `results/` are symlinked to Google Drive (`MyDrive/ATML_PA2_task1`). If the session dies, re-run the setup cells (clone, mount, install, writefile cells) and then continue — finished trainings are detected and skipped.



In [ ]:
!nvidia-smi | head -15
!git clone https://github.com/AbDu11aHHH/ATML-PA2-LLM-PostTraining.git
%cd ATML-PA2-LLM-PostTraining

Thu Oct  8 19:40:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Persist outputs/ and results/ on Google Drive so a timeout doesn't lose adapters
from google.colab import drive
drive.mount('/content/drive')
import os
D='/content/drive/MyDrive/ATML_PA2_task1'
for n in ('outputs','results'):
    os.makedirs(f'{D}/{n}',exist_ok=True)
    if os.path.isdir(n) and not os.path.islink(n): os.system(f'rm -rf {n}')
    if not os.path.islink(n): os.symlink(f'{D}/{n}',n)
!ls -la outputs results; ls outputs/task1_dpo 2>/dev/null

In [ ]:
!pip install -q -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets

## Student code (overwrites starter files)
- **DPO bug fix** in `dpo.py`: logits were `beta*(policy_margin + ref_margin)`; correct is `beta*(policy_margin - ref_margin)`.
- **Over-long prompts:** the starter encoder raises if a prompt alone is ≥ `max_sequence_length` (768). Course data is not edited; such pairs are skipped at load time for every condition, and the skipped source indices are saved in `results/task1_dpo/data_filter_*.json` and the eval JSONs (report these counts).

In [ ]:
%%writefile task1_dpo/dpo.py
from __future__ import annotations

import torch
import torch.nn.functional as F


def dpo_loss(
    policy_chosen_logp: torch.Tensor,
    policy_rejected_logp: torch.Tensor,
    ref_chosen_logp: torch.Tensor,
    ref_rejected_logp: torch.Tensor,
    beta: float,
):
    """Return scalar DPO loss plus lightweight diagnostics.

    Validate this implementation against the equation in the assignment manual
    before using it.
    """

    policy_margin = (
        policy_chosen_logp
        - policy_rejected_logp
    )

    ref_margin = (
        ref_chosen_logp
        - ref_rejected_logp
    )

    # DPO: beta * [(log pi/pi_ref)(y+) - (log pi/pi_ref)(y-)]
    #    = beta * (policy_margin - ref_margin). The starter added ref_margin (bug).
    logits = beta * (
        policy_margin - ref_margin
    )

    loss = -F.logsigmoid(
        logits
    ).mean()

    return loss, {
        "logit_mean":
            logits.detach().mean(),

        "policy_margin_mean":
            policy_margin.detach().mean(),

        "preference_accuracy": (
            (policy_margin - ref_margin) > 0
        ).float().mean().detach(),
    }


In [ ]:
%%writefile task1_dpo/train.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

from common.data import (
    encode_prompt_response,
    load_yaml,
    pad_batch,
    preference_responses,
    prompt_messages_from_preference,
    read_jsonl,
    repo_path,
)
from common.logging_utils import set_seed
from common.generation import response_sequence_logprobs
from common.models import load_policy, load_tokenizer, reference_mode, trainable_parameters
from task1_dpo.dpo import dpo_loss


def make_collate(tokenizer, max_length):
    def collate(rows):
        chosen, rejected = [], []
        for row in rows:
            prompt = prompt_messages_from_preference(row)
            yc, yr = preference_responses(row)
            chosen.append(encode_prompt_response(tokenizer, prompt, yc, max_length))
            rejected.append(encode_prompt_response(tokenizer, prompt, yr, max_length))
        return pad_batch(tokenizer, chosen), pad_batch(tokenizer, rejected)
    return collate


def prompt_token_len(tokenizer, row) -> int:
    ids = tokenizer.apply_chat_template(prompt_messages_from_preference(row), tokenize=True, add_generation_prompt=True)
    if not isinstance(ids, list):  # BatchEncoding in some transformers versions
        ids = ids["input_ids"]
    return len(ids)


def filter_fitting(tokenizer, rows, max_length):
    """The starter encoder raises when the prompt alone >= max_length. Course data must not be edited, so
    over-long-prompt pairs are skipped at load time (identically for every condition) and reported.
    Returns (kept_rows, kept_source_indices, skipped_source_indices)."""
    kept, skipped = [], []
    for i, row in enumerate(rows):
        (kept if prompt_token_len(tokenizer, row) < max_length else skipped).append(i)
    return [rows[i] for i in kept], kept, skipped


def prepare_dpo_run(config_path: str, dataset_path: str | None = None, beta: float | None = None, max_examples: int | None = None):
    cfg = load_yaml(config_path)
    set_seed(int(cfg["seed"]))
    path = dataset_path or cfg["paths"]["dpo_standard_train"]
    tokenizer = load_tokenizer(cfg["base_model"])
    rows, kept_idx, skipped_idx = filter_fitting(tokenizer, read_jsonl(path), int(cfg["max_sequence_length"]))
    if max_examples is not None:  # first n pairs that fit -> same subset for every beta
        rows, kept_idx = rows[: int(max_examples)], kept_idx[: int(max_examples)]
    print(f"[data] {path}: kept {len(rows)} pairs, skipped {len(skipped_idx)} with prompt >= "
          f"{cfg['max_sequence_length']} tokens")
    model = load_policy(cfg, trainable=True, fresh_lora=True)
    loader = DataLoader(
        rows,
        batch_size=int(cfg["batch_size"]),
        shuffle=True,
        collate_fn=make_collate(tokenizer, int(cfg["max_sequence_length"])),
    )
    optimizer = AdamW(
        trainable_parameters(model),
        lr=float(cfg["learning_rate"]),
        weight_decay=float(cfg.get("weight_decay", 0.0)),
    )
    return {
        "cfg": cfg,
        "rows": rows,
        "tokenizer": tokenizer,
        "model": model,
        "loader": loader,
        "optimizer": optimizer,
        "beta": float(cfg["beta"] if beta is None else beta),
        "kept_indices": kept_idx,
        "skipped_indices": skipped_idx,
    }


def run_training(config_path: str, run_name: str, dataset_path: str | None = None, output_path: str | None = None, beta: float | None = None, max_examples: int | None = None):
    bundle = prepare_dpo_run(config_path, dataset_path, beta, max_examples)
    cfg = bundle["cfg"]
    output = repo_path(output_path or cfg["standard_output"])
    output.parent.mkdir(parents=True, exist_ok=True)

    model, loader, opt = bundle["model"], bundle["loader"], bundle["optimizer"]
    beta = bundle["beta"]
    accum = int(cfg["grad_accum_steps"])
    max_norm = float(cfg.get("max_grad_norm", 1.0))
    dev = next(model.parameters()).device
    log_path = repo_path(cfg["results_dir"]) / f"train_log_{run_name}.jsonl"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_f = log_path.open("w")
    json.dump({"dataset": dataset_path or cfg["paths"]["dpo_standard_train"], "n_kept": len(bundle["kept_indices"]),
               "n_skipped_overlong_prompt": len(bundle["skipped_indices"]), "kept_source_indices": bundle["kept_indices"],
               "skipped_source_indices": bundle["skipped_indices"]},
              open(log_path.parent / f"data_filter_{run_name}.json", "w"))

    def to_dev(b):
        return {k: v.to(dev) for k, v in b.items()}

    window, step, n_batches = [], 0, len(loader)
    model.train()
    for it, (chosen, rejected) in enumerate(loader):
        chosen, rejected = to_dev(chosen), to_dev(rejected)
        # frozen reference = same weights with the LoRA adapter disabled
        with torch.no_grad(), reference_mode(model):
            ref_c, _, _ = response_sequence_logprobs(model, chosen)
            ref_r, _, _ = response_sequence_logprobs(model, rejected)
        pol_c, _, _ = response_sequence_logprobs(model, chosen)
        pol_r, _, _ = response_sequence_logprobs(model, rejected)
        loss, diag = dpo_loss(pol_c, pol_r, ref_c, ref_r, beta)
        (loss / accum).backward()
        window.append((float(loss.detach()), float(diag["preference_accuracy"]), float(diag["logit_mean"])))
        if (it + 1) % accum == 0 or it + 1 == n_batches:
            torch.nn.utils.clip_grad_norm_(trainable_parameters(model), max_norm)
            opt.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            rec = {"step": step, "examples_seen": (it + 1) * int(cfg["batch_size"]), "beta": beta,
                   "loss": sum(w[0] for w in window) / len(window),
                   "train_pref_acc": sum(w[1] for w in window) / len(window),
                   "logit_mean": sum(w[2] for w in window) / len(window)}
            window = []
            log_f.write(json.dumps(rec) + "\n"); log_f.flush()
            if step % 5 == 0:
                print(rec, flush=True)
    log_f.close()
    model.save_pretrained(str(output))
    json.dump({"run_name": run_name, "beta": beta, "n_examples": len(bundle["rows"]), "seed": cfg["seed"],
               "dataset": dataset_path or cfg["paths"]["dpo_standard_train"], "lr": cfg["learning_rate"],
               "batch_size": cfg["batch_size"], "grad_accum_steps": accum, "optimizer_steps": step},
              open(output / "run_meta.json", "w"), indent=1)
    print("saved adapter to", output)
    return output


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/dpo.yaml")
    ap.add_argument("--run-name", default="standard")
    ap.add_argument("--dataset")
    ap.add_argument("--output")
    ap.add_argument("--beta", type=float)
    ap.add_argument("--max-examples", type=int)
    args = ap.parse_args()
    run_training(args.config, args.run_name, args.dataset, args.output, args.beta, args.max_examples)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile task1_dpo/evaluate.py
from __future__ import annotations

import argparse
import json

import numpy as np
import torch

from common.data import (encode_prompt_response, load_yaml, pad_batch, preference_responses, prompt_messages,
                         prompt_messages_from_preference, read_jsonl, repo_path)
from common.generation import batch_generate, response_sequence_logprobs, response_token_logprobs, score_reward_pairs
from common.logging_utils import set_seed
from common.metrics import mean_response_length, sampled_kl, word_count, word_limit_compliance
from common.models import clear_gpu, load_policy, load_reward_model, load_tokenizer, reference_mode
from task1_dpo.dpo import dpo_loss
from task1_dpo.train import filter_fitting

STRATUM_KEYS = ("stratum", "length_stratum", "length_bucket", "bucket", "group", "category")


def load_evaluation_bundle(config_path: str, adapter: str, rows_key: str = "dpo_standard_eval", with_reward: bool = True):
    cfg = load_yaml(config_path)
    out = {
        "cfg": cfg,
        "rows": read_jsonl(cfg["paths"][rows_key]),
        "tokenizer": load_tokenizer(cfg["base_model"]),
        "policy": load_policy(cfg, adapter_path=adapter, trainable=False),
    }
    out["reward"] = load_reward_model(cfg) if with_reward else None
    return out


@torch.no_grad()
def preference_stats(policy, tokenizer, rows, cfg, beta, bs=4):
    """Per-pair DPO loss / margin m_theta (sequence logprobs summed over response tokens)."""
    dev = next(policy.parameters()).device
    recs = []
    for i in range(0, len(rows), bs):
        chunk = rows[i:i + bs]
        encs = {"c": [], "r": []}
        for row in chunk:
            pm = prompt_messages_from_preference(row)
            yc, yr = preference_responses(row)
            encs["c"].append(encode_prompt_response(tokenizer, pm, yc, int(cfg["max_sequence_length"])))
            encs["r"].append(encode_prompt_response(tokenizer, pm, yr, int(cfg["max_sequence_length"])))
        bc = {k: v.to(dev) for k, v in pad_batch(tokenizer, encs["c"]).items()}
        br = {k: v.to(dev) for k, v in pad_batch(tokenizer, encs["r"]).items()}
        pc, _, _ = response_sequence_logprobs(policy, bc)
        pr, _, _ = response_sequence_logprobs(policy, br)
        with reference_mode(policy):
            rc, _, _ = response_sequence_logprobs(policy, bc)
            rr, _, _ = response_sequence_logprobs(policy, br)
        m = (pc - rc) - (pr - rr)
        for j in range(len(chunk)):
            loss_j, _ = dpo_loss(pc[j:j+1], pr[j:j+1], rc[j:j+1], rr[j:j+1], beta)
            recs.append({"margin": float(m[j]), "loss": float(loss_j),
                         "len_chosen": int(bc["response_mask"][j].sum()), "len_rejected": int(br["response_mask"][j].sum())})
    return recs


def summarize_pref(recs):
    m = np.array([r["margin"] for r in recs])
    return {"n": len(recs), "dpo_loss": float(np.mean([r["loss"] for r in recs])),
            "pref_acc": float((m > 0).mean()), "margin_mean": float(m.mean())}


@torch.no_grad()
def generate_and_score(bundle, prompts, max_new, gen_bs=16, seed=0):
    """Sample responses; return per-sample response, KL estimate (policy-ref), reward, length."""
    cfg, tok, policy = bundle["cfg"], bundle["tokenizer"], bundle["policy"]
    g = cfg["generation"]
    set_seed(seed)
    out = []
    for i in range(0, len(prompts), gen_bs):
        chunk = prompts[i:i + gen_bs]
        gen = batch_generate(policy, tok, chunk, 1024, max_new, g["temperature"], g["top_p"], g["do_sample"])
        pw, resp, mask = gen["prompt_width"], gen["response_ids"], gen["response_mask"]
        pol_lp, _ = response_token_logprobs(policy, gen["sequences"], gen["attention_mask"], pw, resp)
        with reference_mode(policy):
            ref_lp, _ = response_token_logprobs(policy, gen["sequences"], gen["attention_mask"], pw, resp)
        for j in range(len(chunk)):
            n = int(mask[j].sum())
            # sequence-level sum of log-ratio per sample (course helper = masked token mean; both stored)
            kl_tok = float(sampled_kl(pol_lp[j:j+1], ref_lp[j:j+1], mask[j:j+1]))
            kl_seq = float(((pol_lp[j] - ref_lp[j]) * mask[j]).sum())
            out.append({"response": gen["responses"][j], "length": n, "kl_token_mean": kl_tok, "kl_seq_sum": kl_seq,
                        "truncated": gen["truncated"][j]})
        if bundle.get("reward") is not None:
            rm, rtok = bundle["reward"]
            sc = score_reward_pairs(rm, rtok, chunk, gen["responses"]).cpu().tolist()
            for k, s_ in enumerate(sc):
                out[i + k]["reward"] = float(s_)
    return out


def length_stats(lengths):
    a = np.array(lengths, dtype=float)
    q1, q3 = np.percentile(a, [25, 75])
    return {"len_mean": float(a.mean()), "len_std": float(a.std()), "len_iqr": float(q3 - q1)}


def full_eval(config_path, adapter, name, beta=None, n_gen_prompts=None, max_pref=None):
    bundle = load_evaluation_bundle(config_path, adapter)
    cfg, tok = bundle["cfg"], bundle["tokenizer"]
    beta = float(beta if beta is not None else cfg["beta"])  # beta used for held-out DPO loss
    all_rows = bundle["rows"][: max_pref] if max_pref else bundle["rows"]
    rows, kept_idx, skipped_idx = filter_fitting(tok, all_rows, int(cfg["max_sequence_length"]))
    res = {"name": name, "adapter": adapter, "beta_eval": beta, "seed": cfg["seed"],
           "n_heldout_total": len(all_rows), "n_skipped_overlong_prompt": len(skipped_idx),
           "skipped_source_indices": skipped_idx}
    res.update(summarize_pref(preference_stats(bundle["policy"], tok, rows, cfg, beta)))
    n_gen = int(n_gen_prompts or cfg.get("eval_gen_prompts", 100))
    prompts = [prompt_messages_from_preference(r) for r in rows[:n_gen]]
    gens = generate_and_score(bundle, prompts, int(cfg["max_generation_tokens"]), seed=int(cfg["seed"]))
    res["gen_prompt_indices"] = kept_idx[:len(prompts)]  # source row indices in the held-out file
    res["kl_token_mean"] = float(np.mean([g["kl_token_mean"] for g in gens]))
    res["kl_seq_sum"] = float(np.mean([g["kl_seq_sum"] for g in gens]))
    res["reward_mean"] = float(np.mean([g["reward"] for g in gens]))
    res["truncated_frac"] = float(np.mean([g["truncated"] for g in gens]))
    res.update(length_stats([g["length"] for g in gens]))
    # explicit word-limit compliance on the 10 common prompts
    wl = read_jsonl(cfg["paths"]["word_limit_prompts"])
    wl_gens = generate_and_score({**bundle, "reward": None}, [prompt_messages(r) for r in wl],
                                 int(cfg["max_generation_tokens"]), seed=int(cfg["seed"]))
    comp = []
    for r, g in zip(wl, wl_gens):
        c = word_limit_compliance(prompt_messages(r)[-1]["content"], g["response"])
        g.update({"prompt_id": r["prompt_id"], "words": word_count(g["response"]), "compliant": c})
        comp.append(c)
    res["wordlimit_compliance"] = float(np.mean([c for c in comp if c is not None]))
    res["wordlimit_n"] = sum(c is not None for c in comp)
    res["wordlimit_mean_words"] = float(np.mean([g["words"] for g in wl_gens]))
    out_dir = repo_path(cfg["results_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
    json.dump(res, open(out_dir / f"eval_{name}.json", "w"), indent=1)
    with open(out_dir / f"generations_{name}.jsonl", "w") as f:
        for idx, (p, g) in enumerate(zip(prompts, gens)):
            f.write(json.dumps({"row_index": idx, "prompt": p, **g}, ensure_ascii=False) + "\n")
    with open(out_dir / f"wordlimit_{name}.jsonl", "w") as f:
        for g in wl_gens:
            f.write(json.dumps(g, ensure_ascii=False) + "\n")
    print(json.dumps(res, indent=1))
    clear_gpu(bundle)
    return res


def find_stratum_key(rows):
    """Known names first; otherwise any scalar field with exactly 3 distinct values (the 3 length strata)."""
    for k in STRATUM_KEYS:
        if k in rows[0]:
            return k
    skip = {"prompt", "chosen", "rejected", "messages"}
    cands = []
    for k, v in rows[0].items():
        if k in skip or isinstance(v, (list, dict, float)):
            continue
        vals = {str(r.get(k)) for r in rows}
        if len(vals) == 3:
            cands.append(k)
    if len(cands) >= 1:
        print(f"[stratified_eval] auto-detected stratum field {cands[0]!r} (candidates: {cands}); "
              f"values: {sorted({str(r[cands[0]]) for r in rows})}")
        return cands[0]
    raise KeyError(f"No 3-valued stratum field found; row keys = {list(rows[0])}")


def stratified_eval(config_path, adapter, name, beta=None):
    """Held-out preference accuracy per length stratum (preferred-longer / matched / rejected-longer)."""
    bundle = load_evaluation_bundle(config_path, adapter, rows_key="dpo_length_eval", with_reward=False)
    cfg = bundle["cfg"]
    rows, kept_idx, skipped_idx = filter_fitting(bundle["tokenizer"], bundle["rows"], int(cfg["max_sequence_length"]))
    print(f"[stratified_eval] kept {len(rows)}, skipped {len(skipped_idx)} over-long prompts")
    key = find_stratum_key(rows)
    beta = float(beta if beta is not None else cfg["beta"])
    recs = preference_stats(bundle["policy"], bundle["tokenizer"], rows, cfg, beta)
    res = {"name": name, "stratum_key": key, "overall": summarize_pref(recs), "strata": {},
           "n_skipped_overlong_prompt": len(skipped_idx), "skipped_source_indices": skipped_idx}
    for s in sorted({str(r[key]) for r in rows}):
        sub = [rec for rec, r in zip(recs, rows) if str(r[key]) == s]
        res["strata"][s] = summarize_pref(sub)
    out_dir = repo_path(cfg["results_dir"]); out_dir.mkdir(parents=True, exist_ok=True)
    json.dump(res, open(out_dir / f"stratified_{name}.json", "w"), indent=1)
    print(json.dumps(res, indent=1))
    clear_gpu(bundle)
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/dpo.yaml")
    ap.add_argument("--adapter", required=True)
    ap.add_argument("--name", default="standard")
    ap.add_argument("--beta", type=float, help="beta for held-out DPO loss (default: config beta)")
    ap.add_argument("--n-gen-prompts", type=int)
    args = ap.parse_args()
    full_eval(args.config, args.adapter, args.name, args.beta, args.n_gen_prompts)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile task1_dpo/ablate_beta.py
from __future__ import annotations

import argparse
import json

from common.data import load_yaml, repo_path
from common.models import clear_gpu
from task1_dpo.evaluate import full_eval
from task1_dpo.train import run_training


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/dpo.yaml")
    args = ap.parse_args()
    cfg = load_yaml(args.config)
    n = int(cfg["short_ablation_examples"])
    rows = []
    for beta in cfg["betas"]:
        name = f"beta{beta}"
        out = f"outputs/task1_dpo/{name}"
        # identical data subset (first n rows, fixed order), seed, optimizer, LoRA; only beta changes
        if not repo_path(out).joinpath("adapter_config.json").exists():  # resume after disconnect
            run_training(args.config, name, output_path=out, beta=beta, max_examples=n)
            clear_gpu()
        rows.append(full_eval(args.config, out, name, beta=beta))  # held-out DPO loss uses the run's own beta
        clear_gpu()
    json.dump(rows, open(repo_path(cfg["results_dir"]) / "beta_ablation_summary.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile task1_dpo/analyze_length.py
from __future__ import annotations

import argparse
import json

from common.data import load_yaml, repo_path
from common.models import clear_gpu
from task1_dpo.evaluate import full_eval, stratified_eval
from task1_dpo.train import run_training


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/dpo.yaml")
    ap.add_argument("--skip-train", action="store_true")
    ap.add_argument("--standard-adapter", default=None)
    args = ap.parse_args()
    cfg = load_yaml(args.config)
    std = args.standard_adapter or cfg["standard_output"]
    bal = cfg["length_output"]
    if not args.skip_train and not repo_path(bal).joinpath("adapter_config.json").exists():
        run_training(args.config, "length_balanced", dataset_path=cfg["paths"]["dpo_length_train"], output_path=bal)
        clear_gpu()
    summary = {}
    for name, ad in (("standard", std), ("length_balanced", bal)):
        summary[name] = {"stratified": stratified_eval(args.config, ad, name),
                         "generation": full_eval(args.config, ad, f"{name}_lenstudy")}
        clear_gpu()
    json.dump(summary, open(repo_path(cfg["results_dir"]) / "length_study_summary.json", "w"), indent=1)


if __name__ == "__main__":
    main()


In [ ]:
import os
os.makedirs('tests',exist_ok=True)
open('tests/test_dpo_loss.py','w').write('''import math, torch
from task1_dpo.dpo import dpo_loss

def test_matches_equation():
    pc, pr, rc, rr = (torch.tensor([x]) for x in (-10., -14., -12., -13.))
    beta = 0.1
    loss, d = dpo_loss(pc, pr, rc, rr, beta)
    m = (-10 + 12) - (-14 + 13)          # = 3
    assert math.isclose(float(loss), -math.log(1 / (1 + math.exp(-beta * m))), rel_tol=1e-5)
    assert float(d["preference_accuracy"]) == 1.0

def test_zero_at_init():  # policy == reference -> loss = log 2
    x = torch.tensor([-5., -7.])
    loss, _ = dpo_loss(x, x - 1, x, x - 1, 0.3)
    assert math.isclose(float(loss), math.log(2), rel_tol=1e-5)
''')
if 'eval_gen_prompts' not in open('configs/dpo.yaml').read():
    open('configs/dpo.yaml','a').write('\neval_gen_prompts: 100\n')
!python -m pytest -q tests/test_dpo_loss.py

## 1. Standard DPO (1 epoch, β=0.10)
Skipped automatically if the adapter already exists on Drive.

In [ ]:
import os
if os.path.exists('outputs/task1_dpo/standard/adapter_config.json'):
    print('standard adapter found — skipping training')
else:
    !python -m task1_dpo.train --run-name standard

In [ ]:
import os
assert os.path.exists('outputs/task1_dpo/standard/adapter_config.json'), 'Adapter outputs/task1_dpo/standard missing — run the training cell above first (and check it finished without errors).'
!python -m task1_dpo.evaluate --adapter outputs/task1_dpo/standard --name standard

## 2. β study (0.03 / 0.10 / 0.30, 600 examples each)
Resumable: finished β forks are not retrained.

In [ ]:
!python -m task1_dpo.ablate_beta

In [ ]:
# Timestamped backup of results + run metadata + code snapshot to Drive. Safe to re-run after any step.
import shutil, time, os, glob, subprocess
D='/content/drive/MyDrive/ATML_PA2_task1'
dst=f"{D}/backup_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(dst,exist_ok=True)
if os.path.isdir('results/task1_dpo'):
    shutil.copytree(os.path.realpath('results/task1_dpo'), f'{dst}/results_task1_dpo', dirs_exist_ok=True)
for f in glob.glob('outputs/task1_dpo/*/run_meta.json'):
    name=f.split('/')[-2]; os.makedirs(f'{dst}/run_meta',exist_ok=True); shutil.copy(f,f'{dst}/run_meta/{name}.json')
for d in ('task1_dpo','tests','common'):
    shutil.copytree(d, f'{dst}/code/{d}', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
os.makedirs(f'{dst}/code/configs',exist_ok=True)
for f in glob.glob('configs/*.yaml'): shutil.copy(f,f'{dst}/code/configs/')
print('Backed up to', dst)
print(subprocess.run(['ls','-R',dst],capture_output=True,text=True).stdout[:3000])

## 3. Length-confounding
The length-stratum field is auto-detected (known names, else any field with exactly 3 distinct values); the chosen field and its values are printed — check they are the three strata (preferred-longer / matched / rejected-longer).

In [ ]:
import os
assert os.path.exists('outputs/task1_dpo/standard/adapter_config.json'), 'Adapter outputs/task1_dpo/standard missing — run the training cell above first (and check it finished without errors).'
!python -m task1_dpo.analyze_length --standard-adapter outputs/task1_dpo/standard

## Summary tables

In [ ]:
import json,glob,pandas as pd
rows=[json.load(open(f)) for f in sorted(glob.glob('results/task1_dpo/eval_*.json'))]
cols=['name','n','n_skipped_overlong_prompt','dpo_loss','pref_acc','kl_token_mean','kl_seq_sum','reward_mean','len_mean','len_std','len_iqr','wordlimit_compliance']
display(pd.DataFrame(rows)[cols].round(4))
for f in sorted(glob.glob('results/task1_dpo/stratified_*.json')):
    r=json.load(open(f)); print(r['name'],'| stratum field:',r['stratum_key']); display(pd.DataFrame(r['strata']).T.round(4))

## Backup results to Drive
Re-run after each stage.

In [ ]:
# Timestamped backup of results + run metadata + code snapshot to Drive. Safe to re-run after any step.
import shutil, time, os, glob, subprocess
D='/content/drive/MyDrive/ATML_PA2_task1'
dst=f"{D}/backup_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(dst,exist_ok=True)
if os.path.isdir('results/task1_dpo'):
    shutil.copytree(os.path.realpath('results/task1_dpo'), f'{dst}/results_task1_dpo', dirs_exist_ok=True)
for f in glob.glob('outputs/task1_dpo/*/run_meta.json'):
    name=f.split('/')[-2]; os.makedirs(f'{dst}/run_meta',exist_ok=True); shutil.copy(f,f'{dst}/run_meta/{name}.json')
for d in ('task1_dpo','tests','common'):
    shutil.copytree(d, f'{dst}/code/{d}', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
os.makedirs(f'{dst}/code/configs',exist_ok=True)
for f in glob.glob('configs/*.yaml'): shutil.copy(f,f'{dst}/code/configs/')
print('Backed up to', dst)
print(subprocess.run(['ls','-R',dst],capture_output=True,text=True).stdout[:3000])

## Download results (everything is also already on Drive)

In [ ]:
!zip -qr task1_results.zip results/task1_dpo outputs/task1_dpo/*/run_meta.json
from google.colab import files
files.download('task1_results.zip')